# Big Mart Quiz - Data Exploration

In [ ]:
# Import pandas and load the dataset
import pandas as pd

df = pd.read_csv('Big Mart Dataset.csv')

# Preview the data
df.head()

## Question 4 — How many unique stores (Outlet_Identifier)?

In [ ]:
# nunique() counts the number of distinct values in a column
unique_stores = df['Outlet_Identifier'].nunique()

print("Number of unique stores:", unique_stores)

## Question 5 — Are there issues with Item_Fat_Content?

In [ ]:
# value_counts() shows how many times each unique value appears
# This helps us spot inconsistencies (same meaning, different spellings)
df['Item_Fat_Content'].value_counts()

# You'll see: 'Low Fat', 'LF', 'low fat' all mean the same thing
# And: 'Regular', 'reg' mean the same thing
# This is a data quality / dirty data problem that needs to be cleaned

## Question 6 — Mean item weight for "Baking Goods"?

In [ ]:
# Step 1: Filter the dataframe to only rows where Item_Type is 'Baking Goods'
baking_goods = df[df['Item_Type'] == 'Baking Goods']

# Step 2: Calculate the mean of Item_Weight for that filtered group
mean_weight = baking_goods['Item_Weight'].mean()

# Step 3: Round to 2 decimal places as required
print("Mean item weight for Baking Goods:", round(mean_weight, 2))

## Question 7 — Which item type has the highest median weight?

In [ ]:
# groupby() splits the data by Item_Type
# .median() calculates the median weight for each group
# sort_values(ascending=False) puts the highest at the top
median_by_type = df.groupby('Item_Type')['Item_Weight'].median().sort_values(ascending=False)

print(median_by_type)
print("\nItem type with HIGHEST median weight:", median_by_type.idxmax())

## Question 8 — Which outlet type has the highest average MRP for "Baking Goods"?

In [ ]:
# Step 1: Filter to only Baking Goods rows
baking = df[df['Item_Type'] == 'Baking Goods']

# Step 2: Group by Outlet_Type, take the mean of Item_MRP, sort descending
avg_mrp_by_outlet = baking.groupby('Outlet_Type')['Item_MRP'].mean().sort_values(ascending=False)

print(avg_mrp_by_outlet)
print("\nOutlet type with HIGHEST avg MRP for Baking Goods:", avg_mrp_by_outlet.idxmax())

## Question 9 — Draw a histogram of Item_Outlet_Sales. What is its distribution shape?

In [ ]:
import matplotlib.pyplot as plt

# Draw the histogram
df['Item_Outlet_Sales'].hist(bins=30, edgecolor='black')
plt.title('Distribution of Item Outlet Sales')
plt.xlabel('Item_Outlet_Sales')
plt.ylabel('Frequency')
plt.show()

# Skewness > 0 means right skewed (long tail on the right)
# Skewness < 0 means left skewed
# Skewness near 0 means symmetric
print("Skewness:", round(df['Item_Outlet_Sales'].skew(), 4))
print("Mean:", round(df['Item_Outlet_Sales'].mean(), 2))
print("Median:", round(df['Item_Outlet_Sales'].median(), 2))
print("\nSince Mean > Median and skewness > 1, the distribution is RIGHT SKEWED")

## Question 10 — Draw histograms of Item_Outlet_Sales for ALL item types. Is item type important for forecasting?

In [ ]:
import matplotlib.pyplot as plt

# Get all unique item types
item_types = df['Item_Type'].unique()

# Create a grid of subplots — one histogram per item type
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()  # flatten to loop easily

for i, item in enumerate(sorted(item_types)):
    subset = df[df['Item_Type'] == item]['Item_Outlet_Sales']
    axes[i].hist(subset, bins=20, edgecolor='black', color='steelblue')
    axes[i].set_title(item, fontsize=9)
    axes[i].set_xlabel('Sales', fontsize=7)
    axes[i].set_ylabel('Freq', fontsize=7)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Item_Outlet_Sales by Item Type', fontsize=14)
plt.tight_layout()
plt.show()

# Answer: Yes — different item types have different sales distributions,
# so item type IS important to consider when forecasting outlet sales.

## Question 11 — Is Item_Outlet_Sales right skewed in EACH item type? (True/False)

In [ ]:
# groupby Item_Type, then calculate skewness for each group
# If ALL values are positive, every category is right skewed
skewness_by_type = df.groupby('Item_Type')['Item_Outlet_Sales'].skew().sort_values()

print(skewness_by_type)
print("\nAre ALL item types right skewed (skew > 0)?", (skewness_by_type > 0).all())
# Answer: True — every single Item_Type has positive skewness

## Question 12 — In Grocery Stores only, how many items have 0 visibility?

In [ ]:
# Step 1: Filter to only Grocery Store rows
grocery = df[df['Outlet_Type'] == 'Grocery Store']

# Step 2: Among those, count rows where Item_Visibility == 0
zero_visibility = grocery[grocery['Item_Visibility'] == 0]

print("Number of Grocery Store items with 0 visibility:", len(zero_visibility))

## Question 13 — Which Outlet_Location_Type has the highest median outlet sales? (Use a box plot)

In [ ]:
import matplotlib.pyplot as plt

# Box plot: shows median (middle line), spread, and outliers for each group
# The LINE inside each box = the median
df.boxplot(column='Item_Outlet_Sales', by='Outlet_Location_Type', figsize=(8, 5))
plt.title('Item_Outlet_Sales by Outlet Location Type')
plt.suptitle('')  # removes the default pandas subtitle
plt.xlabel('Outlet Location Type')
plt.ylabel('Item_Outlet_Sales')
plt.show()

# Confirm with exact numbers
median_by_location = df.groupby('Outlet_Location_Type')['Item_Outlet_Sales'].median().sort_values(ascending=False)
print(median_by_location)
print("\nHighest median outlet sales:", median_by_location.idxmax())

## Question 14 — Which Outlet_Location_Type has a more pronounced right tail?

In [ ]:
# A "more pronounced right tail" = higher skewness value
# We can read this from the box plot (longer upper whisker + more outliers on top)
# OR confirm numerically with .skew()

skew_by_tier = df.groupby('Outlet_Location_Type')['Item_Outlet_Sales'].skew().sort_values(ascending=False)
print(skew_by_tier)
print("\nMost right-skewed (longest right tail):", skew_by_tier.idxmax())
# Tier 3 has the highest skewness → most pronounced right tail

## Question 15 — Correlation between Item_Outlet_Sales and Item_MRP

In [ ]:
# .corr() calculates the Pearson correlation coefficient between two columns
# Result ranges from -1 (perfect negative) to +1 (perfect positive)
# 0 means no linear relationship

corr = df['Item_Outlet_Sales'].corr(df['Item_MRP'])
print("Correlation between Item_Outlet_Sales and Item_MRP:", round(corr, 2))
# 0.57 = moderate positive correlation — as MRP goes up, sales tend to go up too

## Question 16 — Is the correlation between Outlet_Sales and Item_MRP positive for EACH product type?

In [ ]:
# groupby Item_Type, then correlate Item_Outlet_Sales with Item_MRP within each group
# .unstack() reshapes the result so we can read it cleanly

corr_by_type = (df.groupby('Item_Type')[['Item_Outlet_Sales', 'Item_MRP']]
                  .corr()
                  .unstack()['Item_MRP']['Item_Outlet_Sales'])

print(corr_by_type.sort_values())
print("\nAre ALL correlations positive?", (corr_by_type > 0).all())
# Answer: Yes — every product type shows a positive correlation between MRP and Sales